# Mistral OCR Test Notebook

Run each cell from top to bottom. The notebook uses the images and PDFs already inside `test/` and sends them to Mistral Cloud OCR through the project's `MistralOCREngine` wrapper.

Before running OCR, make sure `.env.local` contains `MISTRAL_API_KEY=...`.

## 1. Setup

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import time
import sys

from IPython.display import Image, Markdown, display

# Notebook is in notebooks/, so repo root is one folder up.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TEST_DIR = ROOT / "test"
OUTPUT_DIR = ROOT / "output" / "mistral_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Test files folder:", TEST_DIR)
print("Notebook output folder:", OUTPUT_DIR)

## 2. Check Mistral config

In [ ]:
from config import MISTRAL_API_KEY, MISTRAL_OCR_MODEL

print("Mistral OCR model:", MISTRAL_OCR_MODEL)
print("MISTRAL_API_KEY set:", bool(MISTRAL_API_KEY))

if not MISTRAL_API_KEY:
    raise RuntimeError("Add MISTRAL_API_KEY to .env.local before running the OCR cells.")

## 3. List test files

In [ ]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".pdf"}

test_files = sorted(
    p for p in TEST_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
)

for index, path in enumerate(test_files):
    print(f"{index:02d}: {path.relative_to(ROOT)}")

print(f"\nFound {len(test_files)} supported files.")

## 4. Pick one file to process

In [ ]:
# Change this number to test a different file from the list above.
selected_index = 0

selected_file = test_files[selected_index]
print("Selected:", selected_file.relative_to(ROOT))

## 5. Preview selected file

In [ ]:
suffix = selected_file.suffix.lower()

if suffix in {".jpg", ".jpeg", ".png", ".webp"}:
    display(Image(filename=str(selected_file), width=900))
elif suffix == ".pdf":
    import fitz

    doc = fitz.open(selected_file)
    page = doc[0]
    pix = page.get_pixmap(matrix=fitz.Matrix(1.5, 1.5))
    preview_path = OUTPUT_DIR / f"preview_{selected_file.stem}.png"
    pix.save(preview_path)
    doc.close()
    display(Image(filename=str(preview_path), width=900))
else:
    print("No preview available for", suffix)

## 6. Run Mistral Cloud OCR on the selected file

In [ ]:
from ocr.mistral_ocr import MistralOCREngine

engine = MistralOCREngine()


def run_mistral_ocr(path):
    return engine.extract_text(str(path))


start = time.perf_counter()
raw_text = run_mistral_ocr(selected_file)
elapsed = time.perf_counter() - start

print(f"OCR returned {len(raw_text)} characters in {elapsed:.1f}s.")
display(Markdown(raw_text or "**No text returned.**"))

## 7. Save selected OCR text

In [ ]:
safe_name = selected_file.relative_to(TEST_DIR).as_posix().replace("/", "__")
text_path = OUTPUT_DIR / f"{safe_name}.md"
text_path.write_text(raw_text, encoding="utf-8")

print("Saved OCR text to:", text_path)

## 8. Optional: extract structured fields from the OCR text

In [ ]:
# This uses the existing local Ollama extractor. Skip this cell if Ollama is not running.
from ai.extractor import LocalOllamaExtractor

extractor = LocalOllamaExtractor()
extracted = extractor.extract_from_text(
    raw_text,
    source_description=f"Mistral OCR notebook: {selected_file.relative_to(ROOT)}",
)

print(json.dumps(extracted, indent=2, ensure_ascii=False))

## 9. Optional: faster cloud batch over every file in `test/`

In [ ]:
# Run this only when you want to send every supported test file to Mistral Cloud.
# Increase max_workers carefully if you hit rate limits. This can use paid API credits.

max_workers = 4
batch_results = []


def process_one_file(path):
    started = time.perf_counter()
    text = run_mistral_ocr(path)
    elapsed = time.perf_counter() - started
    safe_name = path.relative_to(TEST_DIR).as_posix().replace("/", "__")
    output_path = OUTPUT_DIR / f"{safe_name}.md"
    output_path.write_text(text, encoding="utf-8")
    return {
        "file": str(path.relative_to(ROOT)),
        "chars": len(text),
        "seconds": round(elapsed, 2),
        "output": str(output_path.relative_to(ROOT)),
        "status": "ok",
    }


started_all = time.perf_counter()

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_path = {executor.submit(process_one_file, path): path for path in test_files}

    for index, future in enumerate(as_completed(future_to_path), start=1):
        path = future_to_path[future]
        print(f"[{index}/{len(test_files)}] finished {path.relative_to(ROOT)}")

        try:
            batch_results.append(future.result())
        except Exception as exc:
            batch_results.append({
                "file": str(path.relative_to(ROOT)),
                "chars": 0,
                "seconds": None,
                "output": None,
                "status": f"error: {exc}",
            })

batch_results = sorted(batch_results, key=lambda item: item["file"])
elapsed_all = time.perf_counter() - started_all

summary_path = OUTPUT_DIR / "batch_summary.json"
summary_path.write_text(json.dumps(batch_results, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Processed {len(batch_results)} files in {elapsed_all:.1f}s.")
print("Saved batch summary to:", summary_path)
batch_results